In [1]:
import numpy as np

# Load your 15MHz file
raw = np.fromfile("lte_capture_mac.c8", dtype=np.int8)

# Convert to float and create complex numbers
# We divide by 128 to normalize the 8-bit range to roughly [-1.0, 1.0]
data = raw.astype(np.float32) / 128.0
iq_data = data[0::2] + 1j * data[1::2]

In [3]:
# Since the signal is 3 MHz "down" from your center, we shift it "up"
fs = 15.36e6
t = np.arange(len(iq_data)) / fs
shift_freq = 3.0e6  # The 3 MHz offset
iq_centered = iq_data * np.exp(-1j * 2 * np.pi * shift_freq * t)

In [6]:
from scipy import signal
iq_final = signal.resample(iq_centered, len(iq_centered) * 2)
iq_final.astype(np.complex64).tofile("lte_for_daniel.sigmf-data")

In [7]:
# Generate a negative control
import numpy as np

# Parameters to match Daniel's notebook expectation
fs = 30.72e6
duration = 1.0  # 1 second
num_samples = int(fs * duration)

# Generate random Gaussian noise for I and Q
# Standard deviation of 0.1 keeps it in a "safe" power range
i_channel = np.random.normal(0, 0.1, num_samples)
q_channel = np.random.normal(0, 0.1, num_samples)

# Combine into complex64
blank_signal = (i_channel + 1j * q_channel).astype(np.complex64)

# Save to file
blank_signal.tofile("noise_control.sigmf-data")

print(f"Generated {num_samples} samples of pure noise.")

Generated 30720000 samples of pure noise.
